In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.resource_utils import split_datasets_path
from biked_commons.conditioning import conditioning

from biked_commons.design_evaluation.scoring import *

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\benchmarking\../..\biked_commons\prediction\usability_predictors.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which wil

In [2]:
data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)

In [3]:
evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data.columns)
isobjective = torch.tensor(requirement_types) == 1

In [4]:
def get_condition(idx=0):
    rider_condition = conditioning.sample_riders(10, split="test")
    use_case_condition = conditioning.sample_use_case(10, split="test")
    image_embeddings = conditioning.sample_image_embedding(10, split="test")
    condition = {"Rider": rider_condition[idx], "Use Case": use_case_condition[idx], "Embedding": image_embeddings[idx]}
    return condition


In [5]:
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
class BikeBenchProblem(Problem):
    def __init__(self, data_sample_df, conditioning):
        evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data_sample_df.columns)
        data_sample = data_sample_df.to_numpy()
        
        self.conditioning = conditioning

        self.evaluator=evaluator
        self.requirement_names=requirement_names
        isobjective = torch.tensor(requirement_types) == 1
        self.isobjective = isobjective
        objective_scores, constraint_scores = self.evaluate_fn(data_sample)

        n_var = data_sample.shape[1]
        n_obj = len(objective_scores[0])
        n_ieq_constr = len(constraint_scores[0])
        xl = np.min(data_sample, axis=0)
        xu = np.max(data_sample, axis=0)
        super().__init__(n_var=n_var, n_obj=n_obj, n_ieq_constr=n_ieq_constr, xl=xl, xu=xu)

    def evaluate_fn(self, X):
        X_tens = torch.tensor(X, dtype=torch.float32)
        eval_scores = self.evaluator(X_tens, self.conditioning)
        objective_scores = eval_scores[:, self.isobjective].detach().numpy()
        constraint_scores = eval_scores[:, ~self.isobjective].detach().numpy()
        return objective_scores, constraint_scores

    def _evaluate(self, x, out, *args, **kwargs):
        objective_scores, constraint_scores = self.evaluate_fn(x)
        out["F"] = objective_scores
        out["G"] = constraint_scores

In [6]:
condition = get_condition(0)
problem = BikeBenchProblem(data, condition)

algorithm = NSGA2(pop_size=100)

res = minimize(problem,
               algorithm,
               ('n_gen', 10),
               seed=1,
               verbose=True)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\benchmarking\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)
c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\benchmarking\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use 

n_gen  |  n_eval  | n_nds  |     cv_min    |     cv_avg    |      eps      |   indicator  
     1 |      100 |      1 |  1.5000000000 |  5.534519E+02 |             - |             -
     2 |      200 |      1 |  1.5000000000 |  1.335581E+02 |             - |             -


c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\benchmarking\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)
c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\benchmarking\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use 

     3 |      300 |      1 |  0.000000E+00 |  8.5455953622 |             - |             -
     4 |      400 |      1 |  0.000000E+00 |  1.6011503029 |  0.000000E+00 |             f


c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\benchmarking\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)
c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\benchmarking\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use 

     5 |      500 |      1 |  0.000000E+00 |  1.4567673302 |  0.000000E+00 |             f
     6 |      600 |      2 |  0.000000E+00 |  1.4138021743 |  1.0000000000 |         ideal


c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\benchmarking\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)
c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\benchmarking\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use 

     7 |      700 |      3 |  0.000000E+00 |  1.3805376571 |  0.9995245888 |         ideal
     8 |      800 |      7 |  0.000000E+00 |  1.3066803855 |  0.4268566667 |         ideal


c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\benchmarking\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)
c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\benchmarking\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use 

     9 |      900 |     15 |  0.000000E+00 |  1.1323590463 |  0.4989693581 |         ideal
    10 |     1000 |     28 |  0.000000E+00 |  0.7581473219 |  0.4174007341 |         ideal


In [7]:
result_tens = torch.tensor(res.X, dtype=torch.float32)

In [8]:
main_scorer = construct_scorer(MainScores, StandardEvaluations, data.columns)
detailed_scorer = construct_scorer(DetailedScores, StandardEvaluations, data.columns)

Calculating reference point for scoring functions...


c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\benchmarking\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


Calculating reference point for scoring functions...


c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\benchmarking\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


In [9]:
main_scorer(result_tens, condition)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\benchmarking\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


Hypervolume                     0.259682
Constraint Satisfaction Rate    1.000000
Maximum Mean Discrepancy        0.621299
dtype: float64

In [10]:
detailed_scorer(result_tens, condition)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\benchmarking\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


Min Objective Score: Usability Score - 0 to 1                                                                 0.069374
Min Objective Score: Drag Force                                                                               2.370445
Min Objective Score: Knee Angle Error                                                                         0.000000
Min Objective Score: Hip Angle Error                                                                          6.006924
Min Objective Score: Arm Angle Error                                                                         89.045311
Min Objective Score: Mass                                                                                     5.631468
Min Objective Score: Planar Compliance                                                                        0.000000
Min Objective Score: Transverse Compliance                                                                    4.109135
Min Objective Score: Eccentric Compliance       